# Surrogate Factory — UCFatigue
## Chapter 9. Model Validation
Objectives:
- **9.0** Validate the train/test split quality (voxel tesselation proximity method).
- **9.1** Predict fatigue loads on the test set.
- **9.2** Compute metrics (R², MAE, quantile90).
- **9.2b** KS distribution tests: train vs test residuals.
- **9.3** Validate against requirements defined in SF_1.
- **9.4** Generate scatter and ratio plots.

### 0. Workflow initialisation

In [ ]:
import sys
from pathlib import Path

# add repo root so validationlib is importable
repo_root = str(Path('..').resolve().parent)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow('pipeline_config.yaml')
workflow.resume()

### 9. Model Validation

In [ ]:
workflow.import_metadata(stage_name='SF_9_Model_Validation')

In [ ]:
Train_set = workflow.load_data(workflow.config['job_name'] + '_Train_set.csv')
Val_set   = workflow.load_data(workflow.config['job_name'] + '_Val_set.csv')
Test_set  = workflow.load_data(workflow.config['job_name'] + '_Test_set.csv')
print(f'Train set : {Train_set.shape}')
print(f'Val set   : {Val_set.shape}')
print(f'Test set  : {Test_set.shape}')

#### 9.0 Split Validation
Checks that the train/test split is statistically sound:
- **Residual voxel proportion**: fraction of test points in category combinations not seen in training (target ≤ 0.05).
- **Phacking**: test points too close to training points (data leakage risk).
- **Isolated test**: test points too far from any training point (extrapolation).
- **Chi² p-value**: distribution of train vs test samples per voxel (target ≥ 0.05).

In [ ]:
from model_validation.split_val import split_validation
split_result = split_validation(workflow, Train_set, Test_set)

#### 9.0b Data Coverage — Train / Val / Test
Distribution of **inputs** and **outputs** across the three sets using validationlib.
- **doubleHistogram**: overlaid histograms for density comparison.
- **PCA scatter**: 2-D projection of 8-D input space coloured by set membership.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from validationlib.plots.hist import doubleHistogram
from validationlib.plots.cumu import doublecumulative

ms = workflow.metadata.get_step_data(['metadata', 'Model_Selection'])
inputs  = ms['inputs']
outputs = ms['outputs']

# Exclude categorical columns (strings) — PCA and histograms need numeric data
num_inputs = Train_set[inputs].select_dtypes(include='number').columns.tolist()
print(f'Numeric inputs used for plots: {num_inputs}')

# — Input distribution: Train vs Test (numeric only) —
fig = doubleHistogram(
    Train_set[num_inputs], Test_set[num_inputs],
    x1label='Train', x2label='Test', xlabel='Input features',
    multiPlotsKwargs={'figHsize': 18, 'figAspectRatio': 3}
)
fig.suptitle('Input distribution — Train vs Test', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()
plt.close(fig)

# — Output distribution: Train vs Test —
fig2 = doubleHistogram(
    Train_set[outputs], Test_set[outputs],
    x1label='Train', x2label='Test', xlabel='Outputs',
    multiPlotsKwargs={'figHsize': 18, 'figAspectRatio': 3}
)
fig2.suptitle('Output distribution — Train vs Test', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()
plt.close(fig2)

# — PCA scatter: 3 sets in 2-D input space (numeric inputs only) —
all_num = pd.concat([Train_set[num_inputs], Val_set[num_inputs], Test_set[num_inputs]], axis=0)
pca = PCA(n_components=2)
pca.fit(all_num)
trPC = pca.transform(Train_set[num_inputs])
vlPC = pca.transform(Val_set[num_inputs])
tsPC = pca.transform(Test_set[num_inputs])

fig3, ax = plt.subplots(figsize=(8, 6))
ax.scatter(trPC[:,0], trPC[:,1], s=8,  alpha=0.4, label=f'Train (n={len(Train_set)})',  color='steelblue')
ax.scatter(vlPC[:,0], vlPC[:,1], s=12, alpha=0.6, label=f'Val   (n={len(Val_set)})',    color='orange',    marker='s')
ax.scatter(tsPC[:,0], tsPC[:,1], s=12, alpha=0.6, label=f'Test  (n={len(Test_set)})',   color='tomato',    marker='^')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title(f'Train / Val / Test coverage — PCA of numeric inputs')
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig3)

#### 9.1 Predictions

In [ ]:
from model_validation.prediction import predict
model_output = predict(workflow, Test_set)
train_output = predict(workflow, Train_set)  # needed for distribution tests
model_output.head()

#### 9.2 Metrics

In [ ]:
from model_validation.score import calculate_metrics
metrics = calculate_metrics(workflow, Test_set, model_output)
JSON(metrics)

#### 9.2b Distribution Tests (KS: train vs test residuals)
Kolmogorov-Smirnov test comparing the residual distributions on the **training set** vs the **test set** for each model and output.
- H₀: both distributions are the same.
- p-value ≥ 0.05 → ✓ no significant difference (healthy generalisation).
- p-value < 0.05 → ✗ residuals differ (possible overfitting or distribution shift).

In [ ]:
from model_validation.score import distribution_tests
ks_results = distribution_tests(workflow, Train_set, Test_set, train_output, model_output)

#### 9.2c Residual Distribution Plots
Visual comparison of **train vs test residuals** per output:
- `doubleHistogram`: density overlap.
- `doublecumulative`: CDF overlap — large gap → distribution shift.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from validationlib.plots.hist import doubleHistogram
from validationlib.plots.cumu import doublecumulative

ms = workflow.metadata.get_step_data(['metadata', 'Model_Selection'])
outputs = ms['outputs']
models_info = workflow.metadata.get_step_data(['metadata', 'Model_Training', 'Models'])

for info in models_info:
    label = info['label']

    train_res = pd.DataFrame(
        {col: Train_set[col].values - train_output[f'{label}__{col}'].values for col in outputs}
    )
    test_res = pd.DataFrame(
        {col: Test_set[col].values - model_output[f'{label}__{col}'].values for col in outputs}
    )

    # Histogram
    fig = doubleHistogram(
        train_res, test_res,
        x1label='Train residuals', x2label='Test residuals', xlabel='Outputs',
        multiPlotsKwargs={'figHsize': 18, 'figAspectRatio': 3}
    )
    fig.suptitle(f'{label} — Residual Distribution (Train vs Test)', y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # Cumulative
    fig2 = doublecumulative(
        train_res, test_res,
        label1='Train', label2='Test', xlabel='Residuals',
        multiPlotsKwargs={'figHsize': 18, 'figAspectRatio': 3}
    )
    fig2.suptitle(f'{label} — Cumulative Residuals (Train vs Test)', y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close(fig2)

#### 9.3 Validation against requirements

In [ ]:
from model_validation.validation import validate
validate(workflow, metrics)

#### 9.4 Plots

In [ ]:
%matplotlib inline
from model_validation.visualize import plot
plot(workflow, Test_set, model_output)

### Save

In [ ]:
workflow.save_metadata()